# Описание проекта

Заказчик — кредитный отдел банка. Нужно разобраться, влияет ли семейное положение и количество детей клиента на факт погашения кредита в срок. Входные данные от банка — статистика о платёжеспособности клиентов.

Результаты исследования будут учтены при построении модели кредитного скоринга — специальной системы, которая оценивает способность потенциального заёмщика вернуть кредит банку.

# Описание данных

children — количество детей в семье

days_employed — общий трудовой стаж в днях

dob_years — возраст клиента в годах

education — уровень образования клиента

education_id — идентификатор уровня образования

family_status — семейное положение

family_status_id — идентификатор семейного положения

gender — пол клиента

income_type — тип занятости

debt — имел ли задолженность по возврату кредитов

total_income — ежемесячный доход

purpose — цель получения кредита

In [ ]:
import pandas as pd
import numpy as np
data = pd.read_csv('data.csv')
data

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
...,...,...,...,...,...,...,...,...,...,...,...,...
21520,1,-4529.316663,43,среднее,1,гражданский брак,1,F,компаньон,0,224791.862382,операции с жильем
21521,0,343937.404131,67,среднее,1,женат / замужем,0,F,пенсионер,0,155999.806512,сделка с автомобилем
21522,1,-2113.346888,38,среднее,1,гражданский брак,1,M,сотрудник,1,89672.561153,недвижимость
21523,3,-3112.481705,38,среднее,1,женат / замужем,0,M,сотрудник,1,244093.050500,на покупку своего автомобиля


 # Предобработка данных

## 1. определите и заполните пропущенные значения:
- опишите, какие пропущенные значения вы обнаружили;
- приведите возможные причины появления пропусков в данных;
- объясните, по какому принципу заполнены пропуски;

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


In [ ]:
data.isna().sum()  #узнаем сколько пропусков и где они находятся

,0
children,0
days_employed,2174
dob_years,0
education,0
education_id,0
family_status,0
family_status_id,0
gender,0
income_type,0
debt,0


Пропуски обнаружены в стобцах с трудовым стажем в днях и общим доходом

### Удаление и обработка пропусков и артефактов в days_employed

In [ ]:
data['days_employed'] = data['days_employed'].abs()  #чтобы избавиться от артефактов (отрицательных значений) возьмем модуль
data.sort_values('days_employed', ascending=False).head(2000) #заметим неестественно большие значения
data.loc[data['days_employed'] > 30000, 'days_employed'] = np.nan #заменим эти значения nan

In [ ]:
data['days_employed'] = data['days_employed'].fillna(data['days_employed'].median()) #заполним недостающие значения для безработных и пенсионеров общей медианой по столбцу
median_days = data.groupby('income_type')['days_employed'].median()
median_days

,days_employed
income_type,
безработный,1630.019381
в декрете,3296.759962
госслужащий,2385.358043
компаньон,1630.019381
пенсионер,1630.019381
предприниматель,1075.433732
сотрудник,1630.019381
студент,578.751554


### Удаление и обработка пропусков и артефактов в total_income

In [ ]:
data['total_income'] = data['total_income'].fillna(data['total_income'].median()) #пропуски в столбце с доходом также заполним медианой
data['total_income'].isna().sum() #удостоверимся, что пропусков больше нет

np.int64(0)

## 2.Замените вещественный тип данных на целочисленный:


In [ ]:
data.head(3)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     21525 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      21525 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


Можно заметить два столбца типа float64, которые можно перевести в int64 для оптимизации последующей работы с ними

In [ ]:
data['days_employed'] = data['days_employed'].astype(int)
data['total_income'] = data['total_income'].astype(int)

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   children          21525 non-null  int64 
 1   days_employed     21525 non-null  int64 
 2   dob_years         21525 non-null  int64 
 3   education         21525 non-null  object
 4   education_id      21525 non-null  int64 
 5   family_status     21525 non-null  object
 6   family_status_id  21525 non-null  int64 
 7   gender            21525 non-null  object
 8   income_type       21525 non-null  object
 9   debt              21525 non-null  int64 
 10  total_income      21525 non-null  int64 
 11  purpose           21525 non-null  object
dtypes: int64(7), object(5)
memory usage: 2.0+ MB


## 3.Удалите дубликаты:

- поясните, как выбирали метод для поиска и удаления дубликатов в данных;
приведите возможные причины появления дубликатов;

In [ ]:
text_col = ['education', 'family_status','gender', 'income_type', 'purpose']

for col in text_col:
  data[col] = data[col].str.lower() #сначала унифицируем формат записи данных в столбцах - переведем все в нижний регистр

In [ ]:
data.duplicated().sum() #был найден 71 дубликат

np.int64(71)

In [ ]:
data[data.duplicated(keep=False)].sort_values(by=data.columns.tolist()) #выведем дубликаты на экран, чтобы оценить ситуацию
#дубликаты могли появиться из-за возможных ошибок при выгрузке данных

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
15892,0,1630,23,среднее,1,не женат / не замужем,4,f,сотрудник,0,145017,сделка с подержанным автомобилем
19321,0,1630,23,среднее,1,не женат / не замужем,4,f,сотрудник,0,145017,сделка с подержанным автомобилем
3452,0,1630,29,высшее,0,женат / замужем,0,m,сотрудник,0,145017,покупка жилой недвижимости
18328,0,1630,29,высшее,0,женат / замужем,0,m,сотрудник,0,145017,покупка жилой недвижимости
4216,0,1630,30,среднее,1,женат / замужем,0,m,сотрудник,0,145017,строительство жилой недвижимости
...,...,...,...,...,...,...,...,...,...,...,...,...
9238,2,1630,34,среднее,1,женат / замужем,0,f,сотрудник,0,145017,покупка жилья для сдачи
9013,2,1630,36,высшее,0,женат / замужем,0,f,госслужащий,0,145017,получение образования
14432,2,1630,36,высшее,0,женат / замужем,0,f,госслужащий,0,145017,получение образования
11033,2,1630,39,среднее,1,гражданский брак,1,f,сотрудник,0,145017,сыграть свадьбу


In [ ]:
data = data.drop_duplicates() #удалим дубликаты
data.duplicated().sum() #удостоверимся в том, что дубликатов не осталось

np.int64(0)

## 4. Выделите леммы в значениях столбца с целями получения кредита:

- опишите, как вы проводили лемматизацию целей кредита;

In [ ]:
!pip install pymystem3

In [ ]:
from pymystem3 import Mystem
m = Mystem()

all = ' '.join(data['purpose']) #объединим все в одну большую строку с помощью join(), чтобы затем выделить леммы
lemmas = m.lemmatize(all)

from collections import Counter
print(Counter(lemmas)) #посчитаем частоту каждой леммы


Installing mystem to /root/.local/bin/mystem from http://download.cdn.yandex.net/mystem/mystem-3.1-linux-64bit.tar.gz


Counter({' ': 55023, 'недвижимость': 6351, 'покупка': 5897, 'жилье': 4460, 'автомобиль': 4306, 'образование': 4013, 'с': 2918, 'операция': 2604, 'свадьба': 2324, 'свой': 2230, 'на': 2222, 'строительство': 1878, 'высокий': 1374, 'получение': 1314, 'коммерческий': 1311, 'для': 1289, 'жилой': 1230, 'сделка': 941, 'дополнительный': 906, 'заниматься': 904, 'подержать': 853, 'проведение': 768, 'сыграть': 765, 'сдача': 651, 'семья': 638, 'собственный': 635, 'со': 627, 'ремонт': 607, 'приобретение': 461, 'профильный': 436, 'подержанный': 111, '\n': 1})


## 5. Категоризируйте данные:

- перечислите, какие «словари» вы выделили для этого набора данных, и объясните, почему.

Я выделил два словаря, которые основываются на вопросах для следующего блока.

Для ответа на первые два вопроса следующего блока у нас уже есть все необходимое, но на ответы для этих вопросов категоризованных данных пока что нет:

Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

Как разные цели кредита влияют на его возврат в срок?

Соответственно, сделаем категоризацию 1) по уровню дохода и 2) по целям кредита

### Категоризация для total_income

In [ ]:
data['total_income'].describe().round(0) #прежде чем выделять категории, оценим ситуацию по доходам в нашем датасете, чтобы подогнать категории под имеющиеся данные

,total_income
count,21454.0
mean,165226.0
std,98021.0
min,20667.0
25%,107623.0
50%,145017.0
75%,195813.0
max,2265604.0


In [ ]:
def income_group(income): #используя выделенные выше квартили, создадим категории
  if income <= 107623:
    return 'низкий'
  elif income <= 145017:
    return 'средний'
  elif income <= 195813:
    return 'высокий'
  else:
    return 'очень высокий'

data['income_group'] = data['total_income'].apply(income_group)

data['income_group'].value_counts()


/tmp/ipykernel_3263/188939600.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['income_group'] = data['total_income'].apply(income_group)


,count
income_group,
средний,6415
очень высокий,5364
низкий,5364
высокий,4311


### Категоризация для purpose

In [ ]:
data['lemmas'] = data['purpose'].apply(lambda text: ' '.join(m.lemmatize(text)).strip()) #для выделения целей кредита создадим отдельный столбец с леммами
data

/tmp/ipykernel_3263/499036858.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['lemmas'] = data['purpose'].apply(lambda text: ' '.join(m.lemmatize(text)).strip())


,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose,income_group,lemmas
0,1,8437,42,высшее,0,женат / замужем,0,f,сотрудник,0,253875,покупка жилья,очень высокий,покупка жилье
1,1,4024,36,среднее,1,женат / замужем,0,f,сотрудник,0,112080,приобретение автомобиля,средний,приобретение автомобиль
2,0,5623,33,среднее,1,женат / замужем,0,m,сотрудник,0,145885,покупка жилья,высокий,покупка жилье
3,3,4124,32,среднее,1,женат / замужем,0,m,сотрудник,0,267628,дополнительное образование,очень высокий,дополнительный образование
4,0,1630,53,среднее,1,гражданский брак,1,f,пенсионер,0,158616,сыграть свадьбу,высокий,сыграть свадьба
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21520,1,4529,43,среднее,1,гражданский брак,1,f,компаньон,0,224791,операции с жильем,очень высокий,операция с жилье
21521,0,1630,67,среднее,1,женат / замужем,0,f,пенсионер,0,155999,сделка с автомобилем,высокий,сделка с автомобиль
21522,1,2113,38,среднее,1,гражданский брак,1,m,сотрудник,1,89672,недвижимость,низкий,недвижимость
21523,3,3112,38,среднее,1,женат / замужем,0,m,сотрудник,1,244093,на покупку своего автомобиля,очень высокий,на покупка свой автомобиль


In [ ]:
def purpose_group(purpose):
  if 'жилье' in purpose or 'недвижимость' in purpose: #основываясь на информации о частоте появления тех или иных лемм, выделим наиболее популярные цели кредита
    return 'операции с недвижимостью'
  elif 'автомобиль' in purpose:
    return 'операции с автомобилем'
  elif 'образование' in purpose:
    return 'получение образования'
  elif 'свадьба' in purpose:
    return 'проведение свадьбы'
  else:
    return 'другая цель'

data['purpose_group'] = data['lemmas'].apply(purpose_group)
data

data['purpose_group'].value_counts()


/tmp/ipykernel_3263/68555684.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['purpose_group'] = data['lemmas'].apply(purpose_group)


,count
purpose_group,
операции с недвижимостью,10811
операции с автомобилем,4306
получение образования,4013
проведение свадьбы,2324


# Вопросы

Есть ли зависимость между наличием детей и возвратом кредита в срок?

Есть ли зависимость между семейным положением и возвратом кредита в срок?

Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

Как разные цели кредита влияют на его возврат в срок?

Артефакты в children

In [ ]:
data['children'] = data['children'].abs() #чтобы убрать отрицательные значения
data['children'].value_counts()

/tmp/ipykernel_3263/1333144564.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['children'] = data['children'].abs() #чтобы убрать отрицательные значения


,count
children,
0,14091
1,4855
2,2052
3,330
20,76
4,41
5,9


In [ ]:
data.loc[data['children'] == 20, 'children'] = 2 #кажется была допущена ошибка - 76 семей в которых 20 детей быть не может, заменим это значение на 2 ребенка
data['children'].value_counts()

,count
children,
0,14091
1,4855
2,2128
3,330
4,41
5,9


#### Есть ли зависимость между наличием детей и возвратом кредита в срок?

In [ ]:
kids_debt = data.groupby('children')['debt'].mean() * 100
kids_debt

,debt
children,
0,7.543822
1,9.165808
2,9.492481
3,8.181818
4,9.756098
5,0.000000


In [ ]:
def has_kids(number): #напишем функцию, которая будет определять есть ли дети или нет
  if number > 0:
    return 'есть дети'
  if number == 0:
    return 'нет детей'

data['has_kids'] = data['children'].apply(has_kids)

/tmp/ipykernel_3263/881650117.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['has_kids'] = data['children'].apply(has_kids)


In [ ]:
data.groupby('has_kids')['debt'].mean() * 100

,debt
has_kids,
есть дети,9.208203
нет детей,7.543822


Можно заметить, что существует некоторая зависимость между наличием детей и погашением кредита в срок: люди с детьми чаще допускают просрочки по кредиту, а отсутствие детей связано с большей вероятностью погашения кредита в срок.

#### Есть ли зависимость между семейным положением и возвратом кредита в срок?

In [ ]:
status_debt = data.groupby('family_status')['debt'].mean() * 100
status_debt.round(2)

,debt
family_status,
в разводе,7.11
вдовец / вдова,6.57
гражданский брак,9.35
женат / замужем,7.55
не женат / не замужем,9.75


Здесь подметим, что семейное положение связано с возвратом кредита в срок. Наименьшая доля просрочек наблюдается среди вдовцов/вдов, а также среди тех, кто находится в разводе. Наибольшая доля просрочек характерна для клиентов, не состоящих в браке, а также тех, кто состоит в гражданском браке.

#### Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

In [ ]:
income_debt = data.groupby('income_group')['debt'].mean() * 100
income_debt.round(2)

,debt
income_group,
высокий,8.91
низкий,7.96
очень высокий,7.14
средний,8.53


Здесь можно увидеть, что различия между разным группами невелики. Наименьшая доля просрочек наблюдается у группы с очень высоким доходом, а наибольшая - у людей с высоким доходом. Тем не менее мы не можем выделить четкой закономерности, согласно которой с ростом уровня дохода вероятность просрочки уменьшается. Таким образом, отметим, что некоторая связь есть, но выражена она слабее.

#### Как разные цели кредита влияют на его возврат в срок?

In [ ]:
purpose_debt = data.groupby('purpose_group')['debt'].mean() * 100
purpose_debt.round(2)

,debt
purpose_group,
операции с автомобилем,9.36
операции с недвижимостью,7.23
получение образования,9.22
проведение свадьбы,8.00


Здесь можно увидеть, что цель кредита связана с вероятностью его своевременного возврата. Наименьшая доля просрочек наблюдается по кредитам, оформленным на операции с недвижимостью, а наибольшая - по кредитам, оформленным на операции с автомобилем и получение образования.

In [ ]:
purpose_debt = data.groupby('purpose_group').agg(clients =('debt', 'count'), debt_percent=('debt', 'mean'))
purpose_debt['debt_percent'] = (purpose_debt['debt_percent'] * 100).round(2)
purpose_debt #здесь можно увидеть точное число клиентов, для каждой группы целей кредита

,clients,debt_percent
purpose_group,,
операции с автомобилем,4306,9.36
операции с недвижимостью,10811,7.23
получение образования,4013,9.22
проведение свадьбы,2324,8.00
